# Claim Extraction

This notebook prepares the claim extraction pipeline that sits **before** DeBERTa.

When a user submits any paragraph of text, this module:
1. Splits it into sentences
2. Filters out non-factual sentences (opinions, questions, filler)
3. Uses an LLM to decompose compound sentences into **atomic claims**
4. Deduplicates and returns a clean list ready for DeBERTa

**Choose your mode:**
- `use_llm=False` — fast, offline, heuristics only (good enough for demos)
- `use_llm=True, provider='ollama'` — best quality, local LLM (recommended)
- `use_llm=True, provider='anthropic'` — best quality, Claude API

In [1]:
%pip install spacy -q
# Download the English model (run once)
import subprocess
subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

# If you plan to use Ollama locally:
# %pip install openai -q

# If you plan to use Anthropic Claude:
# %pip install anthropic -q

Note: you may need to restart the kernel to use updated packages.


CompletedProcess(args=['python', '-m', 'spacy', 'download', 'en_core_web_sm'], returncode=0)

In [2]:
import re
import json
import spacy
from typing import Optional

print("Imports OK")

Imports OK


In [3]:
# ─────────────────────────────────────────────────────────────────
# Non-claim filter patterns
# These sentence types are not factual claims and should be discarded
# before we try to verify anything.
# ─────────────────────────────────────────────────────────────────

NON_CLAIM_PATTERNS = [
    r"^\s*(?:who|what|when|where|why|how|is|are|was|were|do|does|did|can|could|will|would|should)\b.+\?",  # questions
    r"^\s*(?:wow|oh|hey|hmm|well|look|note|notice)\b",                                                      # exclamations
    r"\b(?:i think|i believe|i feel|in my opinion|i guess|i reckon|i suppose)\b",                           # opinions
    r"^\s*(?:please|kindly|make sure|ensure|try|consider|remember|note that)\b",                            # imperatives
    r"^\s*(?:however|moreover|furthermore|additionally|therefore|thus|hence|in conclusion|in summary|as a result|for example|for instance|such as|namely)\s*[,.]?\s*$",  # filler
    r"https?://",                                                                                             # URLs
    r"\[\d+\]|\(\d{4}\)",                                                                                    # citations
]

NON_CLAIM_REGEX = [re.compile(p, re.IGNORECASE) for p in NON_CLAIM_PATTERNS]

print(f"{len(NON_CLAIM_PATTERNS)} non-claim patterns loaded")

7 non-claim patterns loaded


In [4]:
class HeuristicExtractor:
    """
    Fast rule-based claim extractor.
    No LLM needed — works entirely offline.
    """

    MIN_WORDS = 5
    MIN_CHARS = 20

    def __init__(self):
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            print("spaCy model not found. Using regex fallback.")
            self.nlp = None

    def extract(self, text: str) -> list:
        sentences = self._split(text)
        return [
            s for s in (self._clean(s) for s in sentences)
            if s
            and len(s) >= self.MIN_CHARS
            and len(s.split()) >= self.MIN_WORDS
            and not self._is_non_claim(s)
        ]

    def _split(self, text):
        if self.nlp:
            return [s.text.strip() for s in self.nlp(text).sents if s.text.strip()]
        return [p.strip() for p in re.split(r'(?<=[.!?])\s+(?=[A-Z])', text) if p.strip()]

    def _clean(self, s):
        s = re.sub(r'\s+', ' ', s).strip()
        s = re.sub(r'^[-*•·▪–—]+\s*', '', s)      # bullet points
        s = re.sub(r'\*\*(.+?)\*\*', r'\1', s)     # **bold**
        s = re.sub(r'`(.+?)`', r'\1', s)            # `code`
        s = re.sub(r'^\d+[.)]\s*', '', s)           # numbered lists
        return s.strip()

    def _is_non_claim(self, s):
        return any(p.search(s) for p in NON_CLAIM_REGEX)


# Quick smoke test
h = HeuristicExtractor()
test = "Einstein was born in 1879. I think he was brilliant. Who was he married to?"
print("Heuristic test:", h.extract(test))

Heuristic test: ['Einstein was born in 1879.']


In [5]:
LLM_SYSTEM_PROMPT = """You are a claim extraction assistant. Break the given text into atomic, self-contained, verifiable factual claims.

Rules:
- Each claim = ONE fact only.
- Each claim must be a complete sentence that can stand alone.
- Remove opinions, questions, rhetorical statements, filler.
- Resolve pronouns (replace he/she/it with the entity name).
- Split compound sentences joined by 'and' into separate claims.
- Preserve exact numbers, dates, proper nouns.
- Do NOT infer or add information not explicitly stated.
- Return ONLY a valid JSON array of strings. No preamble, no markdown.

Example input: "Einstein, born in Germany in 1879, developed relativity and later moved to the US."
Example output: ["Einstein was born in Germany.", "Einstein was born in 1879.", "Einstein developed the theory of relativity.", "Einstein moved to the United States."]"""


class LLMExtractor:
    """
    LLM-based claim extractor. Supports Ollama (local) or Anthropic API.

    Parameters
    ----------
    provider : 'ollama' | 'anthropic' | 'openai'
    model    : model name (None = use provider default)
    fallback : HeuristicExtractor to use if LLM fails
    """

    DEFAULTS = {"ollama": "llama3", "openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}

    def __init__(self, provider="ollama", model=None, api_key=None, fallback=None):
        self.provider = provider
        self.model    = model or self.DEFAULTS.get(provider, "llama3")
        self.fallback = fallback or HeuristicExtractor()
        self.client   = self._build_client(api_key)

    def extract(self, text):
        try:
            raw    = self._call(text)
            claims = self._parse(raw)
            return [c.strip() for c in claims if isinstance(c, str) and c.strip()]
        except Exception as e:
            print(f"[LLMExtractor] Error: {e}  → falling back to heuristics")
            return self.fallback.extract(text)

    def _build_client(self, api_key):
        import os
        if self.provider == "anthropic":
            import anthropic
            return anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        else:
            import openai
            url = "http://localhost:11434/v1" if self.provider == "ollama" else None
            key = api_key or ("ollama" if self.provider == "ollama" else os.environ.get("OPENAI_API_KEY"))
            return openai.OpenAI(api_key=key, base_url=url)

    def _call(self, text):
        user_msg = f"Extract all atomic factual claims:\n\n{text}"
        if self.provider == "anthropic":
            r = self.client.messages.create(
                model=self.model, max_tokens=1024,
                system=LLM_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            return r.content[0].text
        else:
            r = self.client.chat.completions.create(
                model=self.model, temperature=0.0, max_tokens=1024,
                messages=[
                    {"role": "system", "content": LLM_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
            )
            return r.choices[0].message.content

    def _parse(self, raw):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
        m = re.search(r'\[.*?\]', raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
        # Line-by-line fallback
        return [l.strip().strip('"').strip("'").strip('-').strip()
                for l in raw.split('\n') if len(l.strip()) > 10]


print("LLMExtractor class ready.")

LLMExtractor class ready.


In [6]:
class ClaimExtractor:
    """
    Production claim extractor:
      1. Heuristic pre-filter
      2. LLM atomic decomposition (optional)
      3. Deduplication

    Usage:
        extractor = ClaimExtractor(use_llm=False)              # offline
        extractor = ClaimExtractor(use_llm=True,               # Ollama
                                   llm_provider='ollama')
        extractor = ClaimExtractor(use_llm=True,               # Claude
                                   llm_provider='anthropic',
                                   api_key='sk-ant-...')
    """

    def __init__(self, use_llm=True, llm_provider="ollama",
                 llm_model=None, api_key=None, dedup_threshold=0.85):
        self.dedup_threshold = dedup_threshold
        self.heuristic = HeuristicExtractor()
        self.llm = LLMExtractor(
            provider=llm_provider, model=llm_model,
            api_key=api_key, fallback=self.heuristic
        ) if use_llm else None

    def extract(self, text, verbose=False):
        """
        Extract clean atomic factual claims from any input text.

        Returns: list of claim strings
        """
        text = text.strip()
        if not text:
            return []

        # Step 1: heuristic filter
        candidates = self.heuristic.extract(text)
        if verbose:
            print(f"[Heuristic] {len(candidates)} candidates")
            for c in candidates: print(f"  • {c}")
        if not candidates:
            return []

        # Step 2: LLM decomposition
        if self.llm:
            claims = self.llm.extract(" ".join(candidates))
            if verbose:
                print(f"\n[LLM] {len(claims)} atomic claims")
                for c in claims: print(f"  • {c}")
        else:
            claims = candidates

        # Step 3: deduplicate
        unique = self._deduplicate(claims)
        if verbose:
            print(f"\n[Final] {len(unique)} unique claims")
        return unique

    def _jaccard(self, a, b):
        sa, sb = set(a.lower().split()), set(b.lower().split())
        if not sa or not sb: return 0.0
        return len(sa & sb) / len(sa | sb)

    def _deduplicate(self, claims):
        unique = []
        for c in claims:
            if not any(self._jaccard(c, k) >= self.dedup_threshold for k in unique):
                unique.append(c)
        return unique


print("ClaimExtractor class ready.")

ClaimExtractor class ready.


In [7]:
# ─────────────────────────────────────────────────────────────────
# DEMO — test with heuristics only (no LLM needed)
# ─────────────────────────────────────────────────────────────────

extractor = ClaimExtractor(use_llm=False)

test_paragraph = """
Albert Einstein was born in Ulm, Germany in 1879. I think he was one of the
greatest scientists in history. He developed the special theory of relativity
in 1905 and won the Nobel Prize in Physics in 1921. The prize was awarded not
for relativity, but for his discovery of the photoelectric effect. Who do you
think influenced him the most? He later moved to the United States and joined
the Institute for Advanced Study at Princeton.
"""

print("INPUT TEXT:")
print(test_paragraph.strip())
print()

claims = extractor.extract(test_paragraph, verbose=True)

print("\n" + "="*50)
print(f"EXTRACTED {len(claims)} CLAIMS:")
for i, claim in enumerate(claims, 1):
    print(f"  {i}. {claim}")

INPUT TEXT:
Albert Einstein was born in Ulm, Germany in 1879. I think he was one of the
greatest scientists in history. He developed the special theory of relativity
in 1905 and won the Nobel Prize in Physics in 1921. The prize was awarded not
for relativity, but for his discovery of the photoelectric effect. Who do you
think influenced him the most? He later moved to the United States and joined
the Institute for Advanced Study at Princeton.

[Heuristic] 4 candidates
  • Albert Einstein was born in Ulm, Germany in 1879.
  • He developed the special theory of relativity in 1905 and won the Nobel Prize in Physics in 1921.
  • The prize was awarded not for relativity, but for his discovery of the photoelectric effect.
  • He later moved to the United States and joined the Institute for Advanced Study at Princeton.

[Final] 4 unique claims

EXTRACTED 4 CLAIMS:
  1. Albert Einstein was born in Ulm, Germany in 1879.
  2. He developed the special theory of relativity in 1905 and won the Nobe

In [ ]:
# ─────────────────────────────────────────────────────────────────
# OPTIONAL: test with Ollama (local LLM)
#
# Prerequisites:
#   1. Install Ollama: https://ollama.ai
#   2. Pull a model:  ollama pull llama3
#   3. Start server:  ollama serve
#   4. pip install openai
# ─────────────────────────────────────────────────────────────────

# Uncomment when Ollama is running:

# llm_extractor = ClaimExtractor(
#     use_llm=True,
#     llm_provider='ollama',
#     llm_model='llama3',    # or 'mistral', 'gemma2', etc.
# )
#
# compound_sentence = """
#     Marie Curie was a Polish-French physicist and chemist who conducted
#     pioneering research on radioactivity. She was the first woman to win a
#     Nobel Prize and the only person to win Nobel Prizes in two different
#     sciences — Physics in 1903 and Chemistry in 1911.
# """
#
# claims = llm_extractor.extract(compound_sentence, verbose=True)
# print("\nFinal claims:")
# for c in claims:
#     print(f"  → {c}")

print("(LLM demo cell — uncomment and run after setting up Ollama)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# INTEGRATION with DeBERTa pipeline
#
# After you fine-tune DeBERTa, plug the extractor in like this:
# ─────────────────────────────────────────────────────────────────

def analyze_paragraph(paragraph, extractor, deberta_predict_fn):
    """
    Full hallucination analysis pipeline.

    Parameters
    ----------
    paragraph : str
        Raw text from user.
    extractor : ClaimExtractor
        Claim extraction instance.
    deberta_predict_fn : callable
        The predict() function from your DeBERTa notebook.
        Signature: predict(claim, evidence) → {label, confidence, scores}

    Returns
    -------
    list of dicts — one per claim
    """
    claims = extractor.extract(paragraph)

    results = []
    for claim in claims:
        # At inference time, the RAG system retrieves evidence from Wikipedia.
        # For now we use a placeholder — replace with your RAG retrieval.
        retrieved_evidence = "[Retrieved from Wikipedia — implement RAG here]"

        prediction = deberta_predict_fn(claim, retrieved_evidence)

        results.append({
            "claim":      claim,
            "evidence":   retrieved_evidence,
            "label":      prediction["label"],
            "confidence": prediction["confidence"],
            "scores":     prediction["scores"],
        })

    return results


# Example usage (replace `predict` with your actual DeBERTa predict function):
#
# results = analyze_paragraph(
#     paragraph="Einstein won the Nobel Prize in 1921 for relativity.",
#     extractor=extractor,
#     deberta_predict_fn=predict,   # from DeBERTa_FineTuning.ipynb
# )
#
# for r in results:
#     print(f"CLAIM      : {r['claim']}")
#     print(f"LABEL      : {r['label']} ({r['confidence']:.2%} confidence)")
#     print()

print("Integration template ready. Wire up your DeBERTa predict() function when ready.")